# Figure 3 — public-source analysis

This notebook reconstructs Figure 3 directly from the public TRACERx Renal
and CPTAC ccRCC files acquired by `00_Paper_Data_Download.ipynb`.

It contains no archived plotting values and no checks against expected effect
sizes, correlations, confidence intervals, P values, FDRs, or percentiles.
Checks are limited to source integrity, cohort structure, model validity, and
the processed-file contract consumed by the unchanged plotting notebook.


In [20]:
from pathlib import Path
import hashlib
import json
import os
import re

import numpy as np
import pandas as pd
import pyreadr
import statsmodels.api as sm
from scipy.stats import spearmanr
from sklearn.decomposition import PCA

PROJECT_ROOT = Path(
    os.environ.get("MITOXY_PROJECT_ROOT", "/scratch/ar58064/mitoxy")
).resolve()
MANIFEST_PATH = (
    PROJECT_ROOT / "data" / "provenance" /
    "paper_public_source_manifest.csv"
)
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "figure_3_clean"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GENES = ["BAK1", "BAX", "BID", "SLC7A11"]
CORE3 = ["BAK1", "BAX", "BID"]
manifest = pd.read_csv(MANIFEST_PATH)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def source_path(dataset, role):
    match = manifest.loc[
        manifest["dataset"].eq(dataset) & manifest["role"].eq(role)
    ]
    if len(match) != 1:
        raise RuntimeError(
            f"Expected one source for ({dataset!r}, {role!r}); found {len(match)}"
        )
    row = match.iloc[0]
    path = PROJECT_ROOT / str(row["relative_path"])
    if not path.is_file():
        raise FileNotFoundError(path)
    if path.stat().st_size != int(row["size_bytes"]):
        raise RuntimeError(f"Source-size mismatch: {path}")
    if sha256_file(path) != str(row["sha256"]):
        raise RuntimeError(f"Source SHA-256 mismatch: {path}")
    return path


RAW = {
    "tracerx_tpm": source_path("TRACERx Renal", "regional tumor TPM"),
    "tracerx_annotation": source_path("TRACERx Renal", "regional annotation"),
    "tracerx_clinical": source_path("TRACERx Renal", "clinical annotation"),
    "tracerx_supplement": source_path(
        "TRACERx Renal", "evolutionary/genomic supplement"
    ),
    "cptac_rna": source_path("CPTAC ccRCC", "tumor RNA expression"),
    "cptac_protein": source_path(
        "CPTAC ccRCC", "tumor protein abundance"
    ),
    "cptac_clinical": source_path("CPTAC ccRCC", "clinical supplement"),
    "cptac_phospho": source_path(
        "CPTAC ccRCC", "phosphoproteomics package export"
    ),
}

display(pd.DataFrame({"analysis_key": RAW.keys(), "path": RAW.values()}))
print("Public-source integrity audit: PASS")


,analysis_key,path
0,tracerx_tpm,/scratch/ar58064/mitoxy/data/raw/tracerx_renal...
1,tracerx_annotation,/scratch/ar58064/mitoxy/data/raw/tracerx_renal...
2,tracerx_clinical,/scratch/ar58064/mitoxy/data/raw/tracerx_renal...
3,tracerx_supplement,/scratch/ar58064/mitoxy/data/raw/tracerx_renal...
4,cptac_rna,/scratch/ar58064/mitoxy/data/raw/cptac_ccrcc_p...
5,cptac_protein,/scratch/ar58064/mitoxy/data/raw/cptac_ccrcc_p...
6,cptac_clinical,/scratch/ar58064/mitoxy/data/raw/cptac_ccrcc_p...
7,cptac_phospho,/scratch/ar58064/mitoxy/data/raw/cptac_ccrcc_p...


Public-source integrity audit: PASS


In [21]:
def zscore(values):
    values = pd.Series(values, index=getattr(values, "index", None), dtype=float)
    return (values - values.mean()) / values.std(ddof=0)


def parse_grade(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"(?:G(?:RADE)?\s*)?([1-4])", str(value), flags=re.I)
    return float(match.group(1)) if match else np.nan


def parse_stage(value):
    if pd.isna(value):
        return np.nan
    text = str(value).upper().replace("STAGE", " ")
    match = re.search(r"\b(IV|III|II|I)\b", text)
    if match:
        return {"I": 1., "II": 2., "III": 3., "IV": 4.}[match.group(1)]
    match = re.search(r"\b([1-4])\b", text)
    return float(match.group(1)) if match else np.nan


def normalized(text):
    return re.sub(r"[^a-z0-9]+", " ", str(text).lower()).strip()


def find_column(columns, include, exclude=(), required=True):
    include = [normalized(x) for x in include]
    exclude = [normalized(x) for x in exclude]
    candidates = []
    for column in columns:
        name = normalized(column)
        if all(term in name for term in include) and not any(
            term in name for term in exclude
        ):
            candidates.append(column)
    if not candidates:
        if required:
            raise KeyError(
                f"No column matched include={include}, exclude={exclude}. "
                f"Available columns: {list(columns)}"
            )
        return None
    return sorted(candidates, key=lambda x: (len(str(x)), str(x)))[0]


def read_linkedomics(path):
    matrix = pd.read_csv(path, sep="\t", index_col=0, low_memory=False)
    matrix = matrix.apply(pd.to_numeric, errors="coerce")
    matrix.index = [str(x).split("|")[0].strip().upper() for x in matrix.index]
    matrix.columns = [str(x).strip().upper() for x in matrix.columns]
    if matrix.index.duplicated().any():
        matrix = matrix.groupby(level=0, sort=False).mean()
    return matrix


def fit_hc3(data, outcome, predictors, focal):
    fields = [outcome, *predictors]
    complete = data[fields].replace([np.inf, -np.inf], np.nan).dropna()
    design = sm.add_constant(complete[predictors], has_constant="add")
    model = sm.OLS(complete[outcome].astype(float), design.astype(float)).fit(
        cov_type="HC3", use_t=False
    )
    ci = model.conf_int().loc[focal]
    return {
        "n": len(complete), "beta": float(model.params[focal]),
        "se": float(model.bse[focal]), "p_value": float(model.pvalues[focal]),
        "ci_low": float(ci.iloc[0]), "ci_high": float(ci.iloc[1]),
    }


def fit_logit(data, outcome, predictors, focal):
    complete = data[[outcome, *predictors]].replace(
        [np.inf, -np.inf], np.nan
    ).dropna()
    design = sm.add_constant(complete[predictors], has_constant="add")
    model = sm.Logit(
        complete[outcome].astype(float), design.astype(float)
    ).fit(disp=False)
    ci = model.conf_int().loc[focal]
    return {
        "n": len(complete), "beta": float(model.params[focal]),
        "se": float(model.bse[focal]), "p_value": float(model.pvalues[focal]),
        "ci_low": float(ci.iloc[0]), "ci_high": float(ci.iloc[1]),
    }


## TRACERx patient-level reconstruction


In [22]:
expression = pyreadr.read_r(str(RAW["tracerx_tpm"]))[None]
expression.index = expression.index.astype(str).str.upper()
expression.columns = expression.columns.astype(str)
missing = sorted(set(GENES) - set(expression.index))
assert not missing, f"TRACERx expression is missing: {missing}"

annotation = pd.read_csv(RAW["tracerx_annotation"], sep="\t", dtype=str)
sample_column = find_column(annotation.columns, ["sample"])
patient_column = find_column(annotation.columns, ["patient"])
type_column = find_column(annotation.columns, ["type", "collapsed"])

annotation[sample_column] = annotation[sample_column].astype(str).str.strip()
annotation[patient_column] = annotation[patient_column].astype(str).str.strip()
primary_annotation = annotation.loc[
    annotation[type_column].astype(str).str.upper().eq("PRIMARY")
].copy()
primary_samples = [
    sample for sample in primary_annotation[sample_column]
    if sample in expression.columns
]
assert len(primary_samples) > 0

# Figure 3's architecture and component analyses use the public TPM values
# directly before gene-wise standardization. This differs from the log-TPM
# transformation used for the cross-cohort rank analysis in Figure 1.
region_values = (
    expression.loc[GENES, primary_samples]
    .T.apply(pd.to_numeric, errors="coerce")
)
if region_values.isna().any().any():
    raise ValueError("Missing target-gene TPM values in primary regions")
region_z = region_values.apply(zscore, axis=0)
region_z.columns = [f"z_{gene}" for gene in GENES]
region_z["sample_id"] = region_z.index
region_z = region_z.merge(
    primary_annotation[[sample_column, patient_column]],
    left_on="sample_id", right_on=sample_column,
    how="left", validate="one_to_one",
)
region_z = region_z.rename(columns={patient_column: "patient"})

patient = region_z.groupby("patient")[[f"z_{g}" for g in GENES]].mean()
patient.index = patient.index.astype(str)
patient.index.name = "patient"

# Patient-constant annotation fields.
condition_column = find_column(annotation.columns, ["condition"])
patient_annotation = (
    primary_annotation[[patient_column, condition_column]]
    .groupby(patient_column, as_index=False).first()
    .rename(columns={patient_column: "patient", condition_column: "condition"})
)

clinical = pd.read_csv(RAW["tracerx_clinical"], sep="\t", header=1, dtype=str)
clinical.columns = [str(x).strip() for x in clinical.columns]
clinical_patient = find_column(clinical.columns, ["subject"])
clinical_grade = find_column(clinical.columns, ["fuhrman", "grade"])
clinical_stage = find_column(clinical.columns, ["overall", "stage"])
clinical_age = find_column(clinical.columns, ["age"])
clinical_leibovich = find_column(clinical.columns, ["leibovich"])
clinical_table = pd.DataFrame({
    "patient": clinical[clinical_patient].astype(str).str.strip(),
    "grade": clinical[clinical_grade].map(parse_grade),
    "stage": clinical[clinical_stage].map(parse_stage),
    "age": pd.to_numeric(clinical[clinical_age], errors="coerce"),
    "leibovich": pd.to_numeric(clinical[clinical_leibovich], errors="coerce"),
}).drop_duplicates("patient")

patient = (
    patient.reset_index()
    .merge(clinical_table, on="patient", how="left", validate="one_to_one")
    .merge(patient_annotation, on="patient", how="left", validate="one_to_one")
)
patient["punctuated"] = patient["condition"].astype(str).str.casefold().eq(
    "punctuated"
).astype(float)

# These covariates are carried in the public regional annotation. Their exact
# labels differ slightly across bundle releases, so resolve them by meaning.
purity_column = find_column(annotation.columns, ["purity"])
wgi_column = find_column(annotation.columns, ["wgi"], required=False)
if wgi_column is None:
    wgi_column = find_column(annotation.columns, ["weighted", "genome"], required=False)
ith_column = find_column(annotation.columns, ["ith"], required=False)
if ith_column is None:
    ith_column = find_column(annotation.columns, ["intratum", "heterogeneity"])

covariate_rows = primary_annotation[[patient_column, purity_column, wgi_column, ith_column]].copy()
covariate_rows = covariate_rows.rename(columns={
    patient_column: "patient", purity_column: "purity",
    wgi_column: "wgi", ith_column: "ith",
})
for column in ["purity", "wgi", "ith"]:
    covariate_rows[column] = pd.to_numeric(covariate_rows[column], errors="coerce")
covariates = covariate_rows.groupby("patient", as_index=False).mean(numeric_only=True)
patient = patient.merge(covariates, on="patient", how="left", validate="one_to_one")

required_complete = ["grade", "stage", "age", "purity", "condition", "wgi", "ith"]
patient = patient.dropna(subset=required_complete).copy()
assert patient["patient"].is_unique
assert set(patient["punctuated"].unique()).issubset({0., 1.})
print("TRACERx complete-case patients:", len(patient))
display(patient[required_complete].describe(include="all"))


TRACERx complete-case patients: 73


,grade,stage,age,purity,condition,wgi,ith
count,73.000000,73.000000,73.000000,73.000000,73,73.000000,73.000000
unique,NaN,NaN,NaN,NaN,4,NaN,NaN
top,NaN,NaN,NaN,NaN,Other,NaN,NaN
freq,NaN,NaN,NaN,NaN,31,NaN,NaN
mean,3.027397,2.821918,63.328767,0.448575,NaN,0.297906,1.690188
std,0.744845,1.182540,9.248205,0.134582,NaN,0.201328,2.201832
min,1.000000,1.000000,34.000000,0.195000,NaN,0.040520,0.000000
25%,3.000000,2.000000,58.000000,0.350000,NaN,0.131545,0.333333
50%,3.000000,3.000000,64.000000,0.428000,NaN,0.235950,1.142857
75%,4.000000,4.000000,69.000000,0.550000,NaN,0.454675,2.000000


## Panels A, B, D, and E — score architecture and decomposition


In [23]:
gene_columns = [f"z_{gene}" for gene in GENES]
# Standardize the patient-level regional means so every score architecture and
# single-gene model is expressed per patient-level SD.
gene_matrix = patient[gene_columns].apply(zscore, axis=0)
assert gene_matrix.notna().all().all()

pca = PCA(n_components=1, svd_solver="full")
pc1_values = pca.fit_transform(gene_matrix).ravel()
loadings = pca.components_[0].copy()
if loadings.mean() < 0:
    loadings *= -1
    pc1_values *= -1
panel_a = pd.DataFrame({"gene": GENES, "PC1_loading": loadings})

model_data = patient.copy()
for column in gene_columns:
    model_data[column] = gene_matrix[column]

model_data["Equal4"] = gene_matrix.mean(axis=1)
model_data["PCA1_4"] = pc1_values
model_data["BAK1_centered"] = (
    0.5 * gene_matrix["z_BAK1"]
    + 0.5 * gene_matrix[["z_BAX", "z_BID", "z_SLC7A11"]].mean(axis=1)
)
model_data["Apoptotic3"] = gene_matrix[
    ["z_BAK1", "z_BAX", "z_BID"]
].mean(axis=1)
model_data["BAK1_SLC7A11"] = gene_matrix[
    ["z_BAK1", "z_SLC7A11"]
].mean(axis=1)
model_data["BAK1_only"] = gene_matrix["z_BAK1"]
model_data["Rest3"] = gene_matrix[
    ["z_BAX", "z_BID", "z_SLC7A11"]
].mean(axis=1)

score_names = [
    "Equal4", "PCA1_4", "BAK1_centered", "Apoptotic3",
    "BAK1_SLC7A11", "BAK1_only",
]
architecture_rows = []
for score in score_names:
    model_data[f"{score}_sd"] = zscore(model_data[score])
    grade = fit_hc3(
        model_data, "grade", [f"{score}_sd", "purity", "age"],
        f"{score}_sd"
    )
    stage = fit_hc3(
        model_data, "stage", [f"{score}_sd", "purity", "age"],
        f"{score}_sd"
    )
    punct = fit_hc3(
        model_data, f"{score}_sd",
        ["punctuated", "purity", "age", "grade", "stage", "wgi", "ith"],
        "punctuated",
    )
    architecture_rows.append({
        "score": score, "grade_beta": grade["beta"],
        "stage_beta": stage["beta"], "punctuated_beta": punct["beta"],
        "grade_p": grade["p_value"], "stage_p": stage["p_value"],
        "punctuated_p": punct["p_value"],
    })
panel_b = pd.DataFrame(architecture_rows)
panel_b["score_label"] = panel_b["score"].map({
    "Equal4": "Equal4", "PCA1_4": "PCA1",
    "BAK1_centered": "BAK1-centered", "Apoptotic3": "Core3",
    "BAK1_SLC7A11": "BAK1 + SLC7A11", "BAK1_only": "BAK1 only",
})

single_rows = []
for component, column in [
    ("BAK1", "z_BAK1"), ("BAX", "z_BAX"), ("BID", "z_BID"),
    ("SLC7A11", "z_SLC7A11"), ("Equal4", "Equal4_sd"),
]:
    result = fit_hc3(
        model_data, "grade", [column, "purity", "age"], column
    )
    single_rows.append({"component": component, **result})
panel_d = pd.DataFrame(single_rows)

joint_grade = fit_hc3(
    model_data, "grade", ["z_BAK1", "Rest3", "purity", "age"], "z_BAK1"
)
joint_grade_rest = fit_hc3(
    model_data, "grade", ["z_BAK1", "Rest3", "purity", "age"], "Rest3"
)
logit_predictors = [
    "z_BAK1", "Rest3", "purity", "age", "grade", "stage", "wgi", "ith"
]
joint_punct = fit_logit(
    model_data, "punctuated", logit_predictors, "z_BAK1"
)
joint_punct_rest = fit_logit(
    model_data, "punctuated", logit_predictors, "Rest3"
)
panel_e = pd.DataFrame([
    {"endpoint": "Grade", "component": "BAK1", **joint_grade},
    {"endpoint": "Grade", "component": "Rest3", **joint_grade_rest},
    {"endpoint": "Punctuated evolution", "component": "BAK1", **joint_punct},
    {"endpoint": "Punctuated evolution", "component": "Rest3", **joint_punct_rest},
])

for table in [panel_a, panel_b, panel_d, panel_e]:
    numeric = table.select_dtypes(include=[np.number])
    assert np.isfinite(numeric).all().all()
for table in [panel_d, panel_e]:
    assert (table["ci_low"] <= table["beta"]).all()
    assert (table["beta"] <= table["ci_high"]).all()
    assert table["p_value"].between(0, 1).all()

panel_a.to_csv(OUTPUT_DIR / "panel_a_pc1_loadings.csv", index=False)
panel_b.to_csv(OUTPUT_DIR / "panel_b_score_architectures.csv", index=False)
panel_d.to_csv(OUTPUT_DIR / "panel_d_single_gene_grade.csv", index=False)
panel_e.to_csv(OUTPUT_DIR / "panel_e_joint_models.csv", index=False)

display(panel_a)
display(panel_b)
display(panel_d)
display(panel_e)
print("TRACERx structural audit: PASS")


,gene,PC1_loading
0,BAK1,0.469494
1,BAX,0.569554
2,BID,0.603483
3,SLC7A11,0.301648


,score,grade_beta,stage_beta,punctuated_beta,grade_p,stage_p,punctuated_p,score_label
0,Equal4,0.172936,0.310398,0.467954,0.039400,0.033701,0.041285,Equal4
1,PCA1_4,0.173155,0.300864,0.420162,0.025109,0.048305,0.057593,PCA1
2,BAK1_centered,0.201766,0.335847,0.420317,0.004964,0.023892,0.054859,BAK1-centered
3,Apoptotic3,0.180987,0.291222,0.338755,0.016956,0.072678,0.117908,Core3
4,BAK1_SLC7A11,0.154538,0.292817,0.504109,0.340203,0.187890,0.064062,BAK1 + SLC7A11
5,BAK1_only,0.183187,0.274933,0.267161,0.013160,0.051407,0.290132,BAK1 only


,component,n,beta,se,p_value,ci_low,ci_high
0,BAK1,73,0.183187,0.073883,0.013160,0.038378,0.327996
1,BAX,73,0.114199,0.076150,0.133705,-0.035053,0.263451
2,BID,73,0.099009,0.078332,0.206245,-0.054520,0.252537
3,SLC7A11,73,0.042021,0.434526,0.922961,-0.809635,0.893676
4,Equal4,73,0.172936,0.083950,0.039400,0.008396,0.337475


,endpoint,component,n,beta,se,p_value,ci_low,ci_high
0,Grade,BAK1,73,0.161947,0.081826,0.047798,0.001571,0.322323
1,Grade,Rest3,73,0.109503,0.118601,0.355853,-0.122949,0.341956
2,Punctuated evolution,BAK1,73,0.294788,0.362124,0.415615,-0.414963,1.004538
3,Punctuated evolution,Rest3,73,0.735628,0.439783,0.094385,-0.126331,1.597587


TRACERx structural audit: PASS


## Panel C — CPTAC cross-molecular coherence


In [24]:
cptac_rna = read_linkedomics(RAW["cptac_rna"])
cptac_protein = read_linkedomics(RAW["cptac_protein"])
for name, matrix in [("RNA", cptac_rna), ("protein", cptac_protein)]:
    missing = sorted(set(GENES) - set(matrix.index))
    assert not missing, f"CPTAC {name} is missing {missing}"

clinical = pd.read_excel(
    RAW["cptac_clinical"], sheet_name="ccrcc_clinical_characteristics",
    dtype=str,
)
clinical.columns = [str(x).strip() for x in clinical.columns]
histology_column = find_column(clinical.columns, ["histologic", "type"])
clinical = clinical.loc[
    clinical[histology_column].astype(str).str.casefold().eq(
        "clear cell renal cell carcinoma"
    )
].copy()

rna_ids = set(cptac_rna.columns)
id_column = max(
    clinical.columns,
    key=lambda column: len(
        set(clinical[column].dropna().astype(str).str.strip().str.upper()) & rna_ids
    ),
)
cptac_ids = [
    value for value in clinical[id_column].astype(str).str.strip().str.upper()
    if value in cptac_rna.columns and value in cptac_protein.columns
]
assert len(cptac_ids) == len(set(cptac_ids))

rna = cptac_rna.loc[GENES, cptac_ids].T
protein = cptac_protein.loc[GENES, cptac_ids].T
rna_z = rna.apply(zscore, axis=0)
protein_z = protein.apply(zscore, axis=0)
rna_core3 = rna_z[CORE3].mean(axis=1)
protein_core3 = protein_z[CORE3].mean(axis=1)

phospho = pd.read_pickle(RAW["cptac_phospho"], compression="gzip")
if set(cptac_ids).issubset(set(phospho.index.astype(str).str.upper())):
    phospho.index = phospho.index.astype(str).str.upper()
    phospho_samples = phospho
elif set(cptac_ids).issubset(set(phospho.columns.astype(str).str.upper())):
    phospho.columns = phospho.columns.astype(str).str.upper()
    phospho_samples = phospho.T
else:
    raise RuntimeError("Could not align CPTAC phosphoproteomics sample axis")

def phospho_label(column):
    if isinstance(column, tuple):
        return " ".join(map(str, column)).upper()
    return str(column).upper()

site_columns = [
    column for column in phospho_samples.columns
    if "NDRG1" in phospho_label(column)
    and re.search(r"(?:^|[^0-9])T?366(?:[^0-9]|$)", phospho_label(column))
]
if not site_columns:
    raise KeyError("No NDRG1 T366 phosphosite was found in the public CPTAC export")
ndrg1_t366 = phospho_samples.loc[cptac_ids, site_columns].apply(
    pd.to_numeric, errors="coerce"
).mean(axis=1)

comparison_data = [
    ("RNA vs protein Mitoxy", rna_core3, protein_core3),
    ("SLC7A11 RNA vs BID RNA", rna["SLC7A11"], rna["BID"]),
    ("SLC7A11 RNA vs BID protein", rna["SLC7A11"], protein["BID"]),
    ("Raw NDRG1 phospho vs Mitoxy", ndrg1_t366, rna_core3),
]
rows = []
for comparison, x, y in comparison_data:
    complete = pd.concat([x.rename("x"), y.rename("y")], axis=1).dropna()
    rho, p_value = spearmanr(complete["x"], complete["y"])
    z_value = np.arctanh(np.clip(rho, -0.999999, 0.999999))
    z_se = 1 / np.sqrt(len(complete) - 3)
    rows.append({
        "comparison": comparison, "n": len(complete), "rho": float(rho),
        "ci_low": float(np.tanh(z_value - 1.95996398454 * z_se)),
        "ci_high": float(np.tanh(z_value + 1.95996398454 * z_se)),
        "p_value": float(p_value),
    })
panel_c = pd.DataFrame(rows)

assert np.isfinite(panel_c[["rho", "ci_low", "ci_high", "p_value"]]).all().all()
assert (panel_c["n"] > 3).all()
assert (panel_c["ci_low"] <= panel_c["rho"]).all()
assert (panel_c["rho"] <= panel_c["ci_high"]).all()
assert panel_c["p_value"].between(0, 1).all()
panel_c.to_csv(OUTPUT_DIR / "panel_c_cptac_multiomics.csv", index=False)
display(panel_c)
print("CPTAC structural audit: PASS")


,comparison,n,rho,ci_low,ci_high,p_value
0,RNA vs protein Mitoxy,103,0.622095,0.487228,0.727978,2.317800e-12
1,SLC7A11 RNA vs BID RNA,103,0.451270,0.282408,0.593006,1.717898e-06
2,SLC7A11 RNA vs BID protein,103,0.402858,0.227033,0.553249,2.455228e-05
3,Raw NDRG1 phospho vs Mitoxy,99,0.113816,-0.085518,0.304388,2.619819e-01


CPTAC structural audit: PASS


## Processed-file handoff


In [25]:
required_outputs = {
    "panel_a_pc1_loadings.csv": ["gene", "PC1_loading"],
    "panel_b_score_architectures.csv": [
        "score", "grade_beta", "stage_beta", "punctuated_beta", "score_label"
    ],
    "panel_c_cptac_multiomics.csv": [
        "comparison", "n", "rho", "ci_low", "ci_high", "p_value"
    ],
    "panel_d_single_gene_grade.csv": [
        "component", "n", "beta", "se", "ci_low", "ci_high", "p_value"
    ],
    "panel_e_joint_models.csv": [
        "endpoint", "component", "n", "beta", "se", "ci_low", "ci_high", "p_value"
    ],
}

output_rows = []
for filename, columns in required_outputs.items():
    path = OUTPUT_DIR / filename
    table = pd.read_csv(path)
    missing = sorted(set(columns) - set(table.columns))
    if missing:
        raise KeyError(f"{filename} is missing {missing}")
    output_rows.append({
        "file": filename, "rows": len(table), "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })

processed_manifest = pd.DataFrame(output_rows)
processed_manifest.to_csv(
    OUTPUT_DIR / "figure_3_processed_manifest.csv", index=False
)
display(processed_manifest)
print("Figure 3 processed-file contract: PASS")


,file,rows,size_bytes,sha256
0,panel_a_pc1_loadings.csv,4,115,92c8d69563d1a090b1a836ec170fa271824da827b36222...
1,panel_b_score_architectures.csv,6,917,4d560d89e57ca09377a617b795f2b169118623e828ae5b...
2,panel_c_cptac_multiomics.csv,4,471,9c5d05cfaadfb65c621c142ccd3dc37b1565a31fc1957f...
3,panel_d_single_gene_grade.csv,5,583,549eca5a49d46a571933bb7cb015dfec398d0bf21e7e8a...
4,panel_e_joint_models.csv,4,533,51873747923a1a4a0d4db5ba7bcb26d26358a6851811d5...


Figure 3 processed-file contract: PASS
